---
title: "From Business Information to Analysis-Ready Data"
subtitle: "Loading and extracting data"
jupyter: data-extraction-lesson
---

In [2]:
from pathlib import Path
import os

# Live execution may start in lessons/. The demonstrations use project-root paths.
if not Path("data/raw/customers.csv").exists():
    os.chdir(Path("..").resolve())

print("Working directory:", Path.cwd())

Working directory: /Users/heuchennecedric/Documents/data-extraction-lesson


## Learning objectives

By the end of this lesson you should be able to:

1. Explain how everyday business information differs from structured analytical data.
2. Distinguish structured, semi-structured and unstructured information with business examples.
3. Describe ERP and CRM systems as *sources* of information, not as file formats.
4. Outline how organisations extract authorised subsets of data for analysis.
5. Load CSV, Excel, JSON, SQLite and API data into a pandas DataFrame.
6. Choose the right analytical unit when flattening nested records.
7. Explain why extracting characters from a PDF is not the same as recovering reliable structured fields.

The students in this course already understand data and statistics reasonably well. The Python demonstrations are written for people who are still new to the language. Important commands are explained the first time they appear.

## The HarborLink scenario

All examples in this lesson belong to one fictitious company: **HarborLink**.

HarborLink sells a subscription-based customer-communication platform to small and mid-sized businesses. A customer signs up for a monthly plan, receives invoices, pays them, writes to support, and sometimes complains. HarborLink staff record notes in a CRM. Finance staff monitor unpaid invoices.

The company is invented. Every customer name, email, invoice and identifier in the repository is synthetic. We use one coherent story so that the same people and amounts reappear in the CSV file, the Excel workbook, the JSON file, the database, the API and the PDF invoices.

The analytical question that motivates the lesson is simple:

> Which customers have unpaid or overdue subscriptions, and what do we actually know about them?

Before we can answer that question with statistics or machine learning, we must obtain reliable structured data.

## The pipeline we will keep returning to

```{mermaid}
flowchart LR
    A[Real-world business information] --> B[Sources and documents]
    B --> C[Extraction and retrieval]
    C --> D[Structured observations and variables]
    D --> E[pandas DataFrame]
    E --> F[Exploratory analysis]
    F --> G[Numerical representation]
    G --> H[Statistical or ML algorithms]
```

This lesson concentrates on the first half of the pipeline: from business information to a reliable table. Exploratory analysis and visualisation are developed in **Chapter 3**, after the invoice-level table has been assembled.

## Part 1 — From real-world business information to structured data

### Start with familiar business information

Do not start from file formats. Start from the information that professionals already meet at work.

Around 5 September 2026, HarborLink receives this email from Elena Vargas, customer 1827, who manages Atelier Vega:

> "Hello, I am contacting you because my September subscription has still not been paid. Could you please check?"

A human reader understands the message immediately. The customer is worried about a payment. The month is September. The tone is polite. None of that is yet a statistical observation.

The same business event can leave several traces:

| Everyday situation | Original information | What can become structured fields |
| --- | --- | --- |
| A customer subscribes | Signed contract, welcome email, CRM form | Customer identifier, plan, amount, start date, country |
| An invoice is issued | PDF invoice, entry in the finance system | Invoice identifier, dates, amount, due date, status |
| A payment is overdue | Reminder letter, accounting status | Status, due date, days late, reminder date |
| A customer sends an email | The email itself | Date, channel, customer identifier, a short type, the text |
| A CRM note is typed | Free-text note | Author, date, customer, a coded category, the text |
| A ticket or complaint arrives | Support tool, email, phone summary | Ticket identifier, type, channel, status, text |
| A reminder is sent | Email or letter generated by a process | Date, template, invoice identifier, amount |
| An order is placed | Order form or ERP document | Order identifier, products, quantity, amount |
| A supplier invoice arrives | PDF or e-invoice from a supplier | Supplier, amount, account, due date |

For each example, three layers should be kept apart:

1. The **original business information** — the email, the PDF, the CRM note, the ERP screen.
2. The **structured fields** that can be stored in a table — identifiers, dates, amounts, statuses, channels.
3. The information that may **later** be turned into numerical variables — for example a late-payment indicator, a text embedding, or a sentiment score.

The email above is not automatically a convenient row in a dataset. We can, however, represent the interaction as a structured record:

| customer_id | date | channel | interaction_type | text | amount | status |
| ---: | --- | --- | --- | --- | ---: | --- |
| 1827 | 2026-09-05 | email | payment_issue | Hello, I am contacting you... | 49.90 | unpaid |

In [2]:
import pandas as pd

email_record = pd.DataFrame(
    [
        {
            "customer_id": 1827,
            "date": "2026-09-05",
            "channel": "email",
            "interaction_type": "payment_issue",
            "text": (
                "Hello, I am contacting you because my September "
                "subscription has still not been paid. Could you please check?"
            ),
            "amount": 49.90,
            "status": "unpaid",
        }
    ]
)
email_record

,customer_id,date,channel,interaction_type,text,amount,status
0,1827,2026-09-05,email,payment_issue,"Hello, I am contacting you because my Septembe...",49.9,unpaid


Read this table slowly.

- `customer_id`, `date`, `channel` and `status` are structured fields. They can be filtered, counted and joined to other tables.
- `text` preserves the original message. We do not throw the email away just because we have extracted a few fields.
- `amount` does **not** appear in the email. The number 49.90 comes from another source: the September invoice in HarborLink's finance records. A processing system that guessed "49.90" from the email alone would be inventing a value.
- The structured record can be stored in a table and then loaded into Python.
- The text itself may later be analysed with natural language processing or turned into a numerical representation. That is a later step, not the step we take now.

Three kinds of information are therefore mixed in any real extraction task:

- **Existing metadata** — the email date, the sender, the customer identifier already known to the CRM.
- **Information explicitly present in the text** — "September", "subscription", "not been paid".
- **Information inferred by a processing system** — `interaction_type = payment_issue`, or an amount copied from another system.

Do not pretend that every field can always be extracted perfectly from an email. Extraction creates hypotheses that often need validation.

### Structured, semi-structured and unstructured data

These three labels are pedagogical simplifications. Real business data often sits on a continuum, and one document can contain both a clean table and a free-text comment. The labels remain useful if we treat them as a map, not as a law.

#### Structured data

Structured data is organised into well-defined fields and records. A row is an observation. A column is a variable.

Typical HarborLink examples:

- a customer table;
- a payment table;
- an invoice table;
- transaction records;
- a spreadsheet whose columns are consistent from the first row to the last.

If finance exports a customer list with one row per customer and the same columns on every row, the export is structured.

#### Semi-structured data

Semi-structured data has organisation, but not necessarily the shape of a simple rectangle.

Typical examples:

- JSON objects;
- nested API responses;
- application events;
- logs;
- records that contain lists or objects inside other objects.

A HarborLink interaction document may contain a customer identifier, a nested subscription object, and a *list* of contacts. That is structured enough to parse, but it is not yet one tidy table.

#### Unstructured or weakly structured information

Here the useful structure is not directly represented as a table.

Typical examples:

- emails;
- CRM notes;
- complaints;
- contracts;
- PDF documents;
- scanned invoices;
- free-text customer-service tickets.

The HarborLink email is weakly structured: it has a sender and a date, but the meaning lives in the sentences. A scanned supplier invoice may look perfectly tabular to a human and still be only an image to a computer.

### ERP systems as business information sources

An **ERP** — *Enterprise Resource Planning* — is an integrated business-management information system.

That sentence is easy to rush past, so we unpack it.

- An ERP is **not** a file format. It is not "the SAP file" or "the Oracle CSV".
- An ERP is **not** necessarily one single database. It generally relies on databases, applications, business rules and interfaces.
- It can manage information and processes such as finance, accounting, purchasing, sales, invoices, payments, inventory, suppliers and operations.

HarborLink might use an ERP to keep:

- customer or business-partner information;
- invoices;
- payment records;
- accounting information;
- orders;
- products or services;
- suppliers;
- operational transactions.

An analyst generally does **not** need, and should not automatically manipulate, the entire internal ERP data model. The usual objective is to obtain a **relevant and authorised subset** for a specific analytical task.

HarborLink wants to study late payments. The useful extracted subset might contain:

- invoice identifier;
- customer identifier;
- invoice date;
- due date;
- amount;
- payment status;
- payment date;
- order identifier.

That subset can then be transferred to an analytical system or loaded into Python. Everything else in the ERP — warehouse movements, supplier master data, posting rules — can remain untouched.

### ERP extraction and analytical integration

Enterprises rarely ask a data scientist to click through ERP screens. They use automated mechanisms to extract data for statistical analysis and machine learning.

At a high level, the landscape includes:

- manual or scheduled exports;
- APIs;
- ERP connectors;
- ETL and ELT pipelines;
- data integration platforms;
- replication;
- change data capture, where the product supports it;
- data warehouses;
- lakehouses;
- analytical databases.

The conceptual path is stable even when the products change:

```{mermaid}
flowchart LR
    A[ERP source system] --> B[Extraction or integration]
    B --> C[Analytical data platform]
    C --> D[Python or another analytical tool]
```

**ETL** means *extract, transform, load*: data are cleaned on the way to the analytical store. **ELT** means *extract, load, transform*: data land first, then they are transformed inside the analytical platform. Both patterns exist. The lesson does not require you to choose one.

The exact mechanism depends on the ERP product, the version and deployment model, the available interfaces, licences, authorisations, security policies and the company's data architecture.

Direct access to the ERP's internal database is **not** always available, and it is **not** always recommended. Production systems have business rules, audit trails and access controls. An unauthorised query can be a security incident as well as a technical mistake.

### Examples of ERP systems

The objective of this subsection is awareness, not product training. Companies use integrated business systems. Those systems are important sources of analytical data. We are not learning to configure them.

Common examples include:

- **SAP**, including **SAP S/4HANA**;
- **Oracle ERP** / Oracle Fusion Cloud ERP;
- **Microsoft Dynamics 365**;
- **Odoo**.

Sage, Infor or NetSuite appear in the same landscape. A long catalogue is not useful here.

#### SAP as a concrete illustration

SAP is used here only as an illustration of enterprise extraction. The Python demonstrations do **not** require a SAP installation or access to any real ERP.

In relevant SAP environments, authorised users and integration tools can expose selected business data through views or interfaces. Names you may meet later in a company include:

- **CDS Views** (*Core Data Services Views*) — defined views over business data, intended for consumption rather than for browsing internal tables;
- **SAP BW/4HANA** — an analytical destination that can receive extracted data;
- APIs and other integration interfaces;
- full extraction or incremental / delta extraction, when the landscape supports it.

We will not teach ABAP, SAP configuration, SAP authorisations or the internal SAP data model. The diagram that matters is:

```{mermaid}
flowchart LR
    A[SAP or other ERP] --> B[Authorised interface or connector]
    B --> C[Selected business data]
    C --> D[Analytical platform or files]
    D --> E[Python / pandas]
```

If HarborLink used SAP, the late-payment subset listed above would travel along that path. In this course, the same subset is already waiting for us as a small synthetic database and a few files.

### CRM and other sources

A **CRM** — *Customer Relationship Management* — system is another major source of business information. It typically holds:

- customer profiles;
- customer interactions;
- emails;
- CRM notes;
- sales opportunities;
- customer-service tickets;
- complaints;
- reminders;
- subscriptions;
- contact history.

The conceptual distinction is:

- **ERP**: broad business-management processes such as finance, accounting, purchasing, inventory and operations.
- **CRM**: customer relationships, sales and customer interactions.

A real company often uses several systems at once. HarborLink's unpaid invoice lives in the finance/ERP world. Elena Vargas's email and the staff note live in the CRM world. A data scientist who studies late payments may need both: the amount and due date from finance, and the complaint or reminder history from the CRM.

Other sources sit beside these two systems: Excel files emailed by a colleague, a questionnaire, a web page, a PDF contract, a scanned document, or a JSON payload received through an API.

### The distinctions that close Part 1

Keep these pairs separate. They are the conceptual backbone of the lesson.

| Do not confuse | With |
| --- | --- |
| Business information | Structured data |
| Structured data | Numerical representation |
| Source system (ERP, CRM) | File format (CSV, JSON, PDF) |
| Extraction | Analysis |
| Extraction | Interpretation |
| Original text | Extracted metadata |
| Automatic extraction | Validated information |

**Conclusion of Part 1.** Data science often begins before modelling. It begins by identifying where information exists, understanding its structure, obtaining authorised access to it, and converting it into reliable analytical data.

## Part 2 — Loading and transforming data with Python

The rest of the lesson is practical. Every demonstration uses the HarborLink files in `data/raw/`.

The practical objective is:

**Different sources and representations → Python → structured data → pandas DataFrame**

### What pandas is, and why we use it

A **Python package** is a reusable collection of functions and objects that we install once and then import into a program. `pandas` is the package most often used in Python to store and manipulate tables.

An **import** makes that package available in the current session.

In [3]:
import pandas as pd

- `import pandas` loads the package.
- `as pd` is a conventional short name. After this line, we write `pd` instead of `pandas`.
- `pd` is not a special keyword. It is simply the name we chose to use.

A **DataFrame** is a rectangular in-memory table: rows and columns with labels. In this lesson:

- a **row** is one observation — one customer, one invoice, one interaction, depending on the table;
- a **column** is one variable — `customer_id`, `monthly_amount`, `status`, and so on.

A DataFrame is useful because statistical analysis and most machine-learning tools in Python expect data in this rectangular form. It is not a database. It lives in memory for the current session.

### CSV: a tabular export

A CSV file (*comma-separated values*) is a plain-text representation of a table. Businesses use it constantly: finance exports, CRM extracts, reports sent by email.

HarborLink's customer export is `data/raw/customers.csv`. It is a *denormalised* snapshot: each row already combines customer, subscription and current payment status. That is typical of the files analysts receive. It is not a full ERP model.

In [4]:
customers = pd.read_csv("data/raw/customers.csv")

- `pd` is the pandas package we imported.
- `read_csv` reads a text table and builds a DataFrame.
- The argument `"data/raw/customers.csv"` is the path to the file, relative to the project root.
- The function **returns** a DataFrame. Nothing is saved until we assign it.
- `customers = ...` stores that DataFrame in a variable called `customers`.
- After this line, `customers` represents the HarborLink customer export in memory.

In [5]:
customers.head()

,customer_id,customer_name,company_name,customer_segment,subscription_type,monthly_amount,signup_date,country,payment_status
0,1827,Elena Vargas,Atelier Vega,mid-market,premium,49.9,2025-03-12,BE,unpaid
1,1841,Marc Dubois,Dubois & Fils,smb,standard,29.9,2025-05-04,FR,paid
2,1903,Priya Shah,Shah Consulting,smb,starter,19.9,2025-06-18,NL,paid
3,1918,Thomas Berg,Berg Logistics,mid-market,premium,49.9,2025-01-20,DE,overdue
4,2044,Amelie Laurent,Laurent Design,smb,standard,29.9,2025-09-01,BE,paid


`head()` returns the first five rows. It is an inspection command: we use it to see whether the file looks like the table we expected.

In [6]:
customers.shape

(8, 9)

`shape` is a pair `(n_rows, n_columns)`. Here we have eight customers and nine variables. Small on purpose: the file should be easy to read in class.

In [7]:
customers.columns

Index(['customer_id', 'customer_name', 'company_name', 'customer_segment',
       'subscription_type', 'monthly_amount', 'signup_date', 'country',
       'payment_status'],
      dtype='str')

`columns` lists the variable names. They should match the export we were promised.

In [8]:
customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   customer_id        8 non-null      int64  
 1   customer_name      8 non-null      str    
 2   company_name       8 non-null      str    
 3   customer_segment   8 non-null      str    
 4   subscription_type  8 non-null      str    
 5   monthly_amount     8 non-null      float64
 6   signup_date        8 non-null      str    
 7   country            8 non-null      str    
 8   payment_status     8 non-null      str    
dtypes: float64(1), int64(1), str(7)
memory usage: 708.0 bytes


`info()` summarises the columns, the number of non-missing values and the stored type of each column. Notice that `signup_date` may appear as text (`object`) if we do not ask pandas to parse dates.

The conceptual transition is:

**CSV file → pandas DataFrame → observations and variables**

A few operations are useful as soon as a file is loaded. They are still extraction and preparation, not a full exploratory analysis.

In [9]:
customers = pd.read_csv(
    "data/raw/customers.csv",
    parse_dates=["signup_date"],
)
customers[["customer_id", "signup_date", "payment_status"]].head()

,customer_id,signup_date,payment_status
0,1827,2025-03-12,unpaid
1,1841,2025-05-04,paid
2,1903,2025-06-18,paid
3,1918,2025-01-20,overdue
4,2044,2025-09-01,paid


`parse_dates=["signup_date"]` tells `read_csv` that this column contains dates. The square brackets after `customers` **select columns**. We keep only three variables for display.

In [10]:
unpaid_customers = customers[
    customers["payment_status"].isin(["unpaid", "overdue"])
]
unpaid_customers[["customer_id", "customer_name", "monthly_amount", "payment_status"]]

,customer_id,customer_name,monthly_amount,payment_status
0,1827,Elena Vargas,49.9,unpaid
3,1918,Thomas Berg,49.9,overdue
5,2110,Jonas Keller,19.9,unpaid


The expression inside the square brackets **filters rows**. `isin(...)` keeps the statuses we care about. Elena Vargas is unpaid; Thomas Berg is overdue. Those two words are not interchangeable: an unpaid invoice may still be before its due date.

In [11]:
customers["monthly_amount"].isna().sum()

np.int64(0)

`isna()` marks missing values. `sum()` counts them. This export has none, which is convenient — and not guaranteed in a real company file.

In [12]:
unpaid_customers.to_csv("data/processed/unpaid_customers.csv", index=False)

`to_csv` writes a DataFrame back to disk. `index=False` avoids saving the pandas row numbers as an extra column. The processed file is a new analytical artefact, not a replacement for the source export.

### Excel: the everyday business workbook

Excel is common because people can open it, sort it, add a comment and email it. That convenience is also the problem: a workbook is not automatically a DataFrame.

In [13]:
customers_xlsx = pd.read_excel("data/raw/customers.xlsx")
customers_xlsx.head()

,customer_id,customer_name,company_name,customer_segment,subscription_type,monthly_amount,signup_date,country,payment_status
0,1827,Elena Vargas,Atelier Vega,mid-market,premium,49.9,2025-03-12,BE,unpaid
1,1841,Marc Dubois,Dubois & Fils,smb,standard,29.9,2025-05-04,FR,paid
2,1903,Priya Shah,Shah Consulting,smb,starter,19.9,2025-06-18,NL,paid
3,1918,Thomas Berg,Berg Logistics,mid-market,premium,49.9,2025-01-20,DE,overdue
4,2044,Amelie Laurent,Laurent Design,smb,standard,29.9,2025-09-01,BE,paid


- `read_excel` needs the `openpyxl` package for `.xlsx` files.
- By default it reads the **first worksheet**.
- The result is again a DataFrame. The colours, column widths and merged titles of Excel do not travel with it.

A workbook can contain several sheets. HarborLink's file has a clean customer sheet, a clean invoice sheet, and a notes sheet that was never designed as a table.

In [14]:
excel_file = pd.ExcelFile("data/raw/customers.xlsx")
excel_file.sheet_names

['customers', 'invoices', 'Monthly_notes']

In [15]:
invoices_xlsx = pd.read_excel("data/raw/customers.xlsx", sheet_name="invoices")
invoices_xlsx.head()

,invoice_id,customer_id,subscription_id,order_id,invoice_date,due_date,amount,status,period
0,INV-2026-0801,1827,SUB-1827,ORD-2026-1827-08,2026-08-01,2026-08-15,49.9,paid,August 2026
1,INV-2026-0802,1841,SUB-1841,ORD-2026-1841-08,2026-08-01,2026-08-15,29.9,paid,August 2026
2,INV-2026-0803,1903,SUB-1903,ORD-2026-1903-08,2026-08-01,2026-08-15,19.9,paid,August 2026
3,INV-2026-0804,1918,SUB-1918,ORD-2026-1918-08,2026-08-01,2026-08-15,49.9,paid,August 2026
4,INV-2026-0805,2044,SUB-2044,ORD-2026-2044-08,2026-08-01,2026-08-15,29.9,paid,August 2026


`sheet_name="invoices"` selects a worksheet by name. Programmatic loading is useful because the same command can be re-run next month. A colleague's manual copy-paste cannot.

In [16]:
notes = pd.read_excel("data/raw/customers.xlsx", sheet_name="Monthly_notes", header=None)
notes

,0,1
0,HarborLink — analyst notes (September 2026),NaN
1,NaN,NaN
2,These notes were typed in a spreadsheet. They ...,NaN
3,NaN,NaN
4,Follow-up,Customer 1827 emailed about an unpaid Septembe...
5,Follow-up,Customer 1918 complained after a reminder; Sep...
6,NaN,NaN
7,Do not treat this sheet as analysis-ready data.,NaN


This sheet has a title, empty rows and sentences in a second column. pandas can open it, but the result is not analysis-ready. A spreadsheet may need cleaning or structural interpretation before it becomes a reliable analytical table. Never assume that every Excel workbook is a dataset.

### JSON: objects, lists and nested documents

**JSON** (*JavaScript Object Notation*) is a format for representing structured or semi-structured information.

It is important to say what JSON is *not*:

- JSON is not an API;
- JSON is not a database;
- JSON is not restricted to JavaScript.

Companies use JSON because software systems can produce and consume it easily. It appears in API responses, application events and integration files. It may be stored in a `.json` file or transmitted over a network.

The basic building blocks are:

- an **object**: a collection of named fields, written with braces;
- a **key/value pair**: `"customer_id": 1827`;
- a **list**: an ordered collection, written with square brackets;
- a **nested object**: an object stored inside another object;
- a **nested structure**: lists and objects combined.

HarborLink stores customer interactions as a list of documents in `data/raw/interactions.json`. Each document looks like this:

```json
{
  "customer_id": 1827,
  "subscription": {
    "type": "premium",
    "monthly_amount": 49.90,
    "status": "unpaid"
  },
  "contacts": [
    {
      "date": "2026-09-05",
      "channel": "email",
      "interaction_type": "payment_issue",
      "text": "Hello, I am contacting you because my September subscription has still not been paid. Could you please check?"
    }
  ]
}
```

Python's standard library can read this file. No extra package is required.

In [17]:
import json

with open("data/raw/interactions.json", encoding="utf-8") as file:
    interactions = json.load(file)

type(interactions), len(interactions)

(list, 8)

- `open(...)` opens the file. `encoding="utf-8"` states how characters are stored.
- `with` is a context manager: it closes the file automatically when the block ends, even if something goes wrong.
- `json.load(file)` reads the file and returns a Python object.
- A JSON **object** becomes a Python **dictionary** (`dict`).
- A JSON **list** becomes a Python **list**.
- Nested objects become dictionaries inside dictionaries or lists.

In [18]:
interactions[0]["customer_id"]

1827

In [19]:
interactions[0]["subscription"]

{'type': 'premium', 'monthly_amount': 49.9, 'status': 'unpaid'}

In [20]:
interactions[0]["contacts"][0]["text"]

'Hello, I am contacting you because my September subscription has still not been paid. Could you please check?'

`interactions[0]` is the first customer document. Square brackets select a position in a list or a key in a dictionary. The last line reaches the original email text inside a list of contacts.

A nested document is not yet a DataFrame. The correct flattening depends on the **analytical unit**:

- one row per customer;
- one row per interaction;
- one row per invoice;
- one row per payment.

In [21]:
customers_from_json = pd.json_normalize(interactions)
customers_from_json[
    ["customer_id", "company_name", "subscription.status", "subscription.type"]
]

,customer_id,company_name,subscription.status,subscription.type
0,1827,Atelier Vega,unpaid,premium
1,1841,Dubois & Fils,paid,standard
2,1903,Shah Consulting,paid,starter
3,1918,Berg Logistics,overdue,premium
4,2044,Laurent Design,paid,standard
5,2110,Keller Studio,unpaid,starter
6,2235,Rossi Media,paid,premium
7,2301,Northwind Retail,paid,standard


`pd.json_normalize` turns nested dictionaries into columns. `subscription.status` is the status that lived inside the nested object. Each row is still **one customer**. The contacts remain grouped inside each document; they are not separate rows.

In [22]:
contacts = pd.json_normalize(
    interactions,
    record_path="contacts",
    meta=["customer_id", "company_name", ["subscription", "status"]],
    sep="_",
)
contacts

,date,channel,interaction_type,text,customer_id,company_name,subscription_status
0,2026-09-05,email,payment_issue,"Hello, I am contacting you because my Septembe...",1827,Atelier Vega,unpaid
1,2026-09-06,crm_note,note,Customer emailed about unpaid September premiu...,1827,Atelier Vega,unpaid
2,2026-09-08,reminder,reminder,Friendly reminder: invoice INV-2026-0901 for E...,1827,Atelier Vega,unpaid
3,2026-09-01,crm_note,note,Payment for September received. No follow-up n...,1841,Dubois & Fils,paid
4,2026-09-03,ticket,complaint,We received a reminder although our August inv...,1918,Berg Logistics,overdue
5,2026-09-04,crm_note,note,Complaint about reminder. August invoice is pa...,1918,Berg Logistics,overdue
6,2026-09-02,email,question,Could you send a copy of the September invoice...,2044,Laurent Design,paid
7,2026-09-07,ticket,payment_issue,I think the starter plan was charged twice. Ca...,2110,Keller Studio,unpaid
8,2026-08-20,email,question,We would like to add two extra user seats next...,2235,Rossi Media,paid
9,2026-09-09,ticket,question,Please confirm that our standard plan renews a...,2301,Northwind Retail,paid


- `record_path="contacts"` says: make **one row per contact**.
- `meta=...` copies identifying fields from the parent document onto every contact row.
- `["subscription", "status"]` means: take `status` inside `subscription`.
- One JSON object that contains three contacts becomes three rows.

That is the crucial point. The same file can produce a customer table or an interaction table. The transformation is not unique. It depends on the question. If we study late-payment *customers*, the first table is closer. If we study *complaints and reminders*, the second table is the right unit.

Customers with no contacts disappear from the contact table. That is expected: there is no interaction to turn into a row. It is not a bug in pandas.

### SQL and relational databases

A company uses a database rather than a single CSV file when several tables must stay consistent, when many users update the same records, or when the volume is larger than a convenient spreadsheet.

A **relational database** stores data in tables that can be linked.

- a **table** is a named collection of rows;
- a **row** is a record;
- a **column** is a field;
- a **primary key** uniquely identifies a row (`customer_id` in `customers`);
- a **foreign key** points to a row in another table (`customer_id` in `payments`);
- a **relationship** is that link;
- a **query** asks for a subset;
- **filtering** keeps some rows;
- a **join** combines tables through their keys.

We use **SQLite** because it is a real relational database stored in one file. It does not require an external server. HarborLink's file is `data/raw/company.sqlite`.

In [23]:
import sqlite3

connection = sqlite3.connect("data/raw/company.sqlite")

- `sqlite3` is part of the Python standard library.
- `connect` opens a **connection**: a live link to the database file.
- The database stays on disk. Python does not load every table into memory just by connecting.

In [24]:
query = """
SELECT customer_id, amount, status
FROM payments
WHERE status = 'unpaid'
"""

unpaid_payments = pd.read_sql_query(query, connection)
unpaid_payments

,customer_id,amount,status
0,1827,49.9,unpaid
1,2110,19.9,unpaid


- `SELECT` lists the columns we want.
- `FROM` names the table.
- `WHERE` keeps only the rows that satisfy a condition.
- `pd.read_sql_query` sends the query through the connection and **returns a DataFrame**.
- We retrieve a subset. We do not import the entire database.

A database is not a DataFrame. The database stores and manages data, potentially at large scale, for many processes. The DataFrame is an in-memory structure used for analysis. Python should ask for the relevant subset.

A join brings customer names next to payment problems:

In [25]:
join_query = """
SELECT
    c.customer_id,
    c.customer_name,
    c.company_name,
    p.invoice_id,
    p.amount,
    p.status
FROM payments AS p
JOIN customers AS c
    ON c.customer_id = p.customer_id
WHERE p.status IN ('unpaid', 'overdue')
ORDER BY p.status DESC, c.customer_id
"""

late = pd.read_sql_query(join_query, connection)
late

,customer_id,customer_name,company_name,invoice_id,amount,status
0,1827,Elena Vargas,Atelier Vega,INV-2026-0901,49.9,unpaid
1,2110,Jonas Keller,Keller Studio,INV-2026-0906,19.9,unpaid
2,1918,Thomas Berg,Berg Logistics,INV-2026-0904,49.9,overdue


- `JOIN ... ON` matches each payment to the customer with the same `customer_id`.
- `AS p` and `AS c` are short names for the tables.
- `IN ('unpaid', 'overdue')` keeps both problem statuses.
- The result is again a DataFrame, ready for analysis.

In [26]:
connection.close()

Close the connection when the queries are finished.

The same three HarborLink customers should now feel familiar: Elena Vargas (unpaid, not yet overdue), Jonas Keller (unpaid), Thomas Berg (overdue). The identifiers match the CSV, the Excel invoices and the JSON email.

### APIs: an interface, not a format

An **API** (*Application Programming Interface*) is an interface through which software can access data or services.

The distinction with JSON must stay explicit:

- **JSON** is a format used to represent data.
- **An API** is an interface used to request data or services.

An API may return JSON. It may also return XML, a file or an error message. The format and the interface are different ideas.

The conceptual pipeline is:

```{mermaid}
flowchart LR
    A[Python request] --> B[API endpoint]
    B --> C[Server response]
    C --> D[JSON or another format]
    D --> E[Python object]
    E --> F[pandas DataFrame]
```

A reliable classroom demonstration should not depend on a personal key, a paid service or a website that may change tomorrow. This lesson therefore uses a **local mock API** that serves HarborLink invoices from `data/raw/api_invoices.json`. It reproduces the interaction pattern of a real HTTP API. If the server cannot start, the same JSON file is the offline fallback.

In [27]:
import requests
from python.mock_api import start_background_server

API_URL = start_background_server() + "/invoices"
API_URL

'http://127.0.0.1:8765/invoices'

- `start_background_server()` starts a small local server in the background. If a previous lesson session already left that server running, the helper reuses it.
- An **endpoint** is a URL that names a resource. Here the resource is `/invoices`.
- `requests` is the Python package that sends HTTP requests.

In [28]:
response = requests.get(API_URL)
response.raise_for_status()
response.status_code

200

- `requests.get` sends an **HTTP GET request**: "please give me this resource".
- The **response** is the server's answer. It includes a **status code**.
- `200` means success. Codes in the 400 and 500 ranges indicate a client or server problem.
- `raise_for_status()` stops the program if the request failed. It is safer than ignoring the code.

In [29]:
data = response.json()
data.keys()

dict_keys(['company', 'as_of', 'invoices'])

`response.json()` parses the JSON body into a Python object. Here the object is a dictionary with a company name, a date and a list of invoices.

In [30]:
api_invoices = pd.DataFrame(data["invoices"])
api_invoices

,invoice_id,customer_id,invoice_date,due_date,amount,currency,status,order_id
0,INV-2026-0901,1827,2026-08-28,2026-09-15,49.9,EUR,unpaid,ORD-2026-1827-09
1,INV-2026-0902,1841,2026-08-28,2026-09-15,29.9,EUR,paid,ORD-2026-1841-09
2,INV-2026-0903,1903,2026-08-28,2026-09-15,19.9,EUR,paid,ORD-2026-1903-09
3,INV-2026-0904,1918,2026-08-28,2026-09-01,49.9,EUR,overdue,ORD-2026-1918-09
4,INV-2026-0905,2044,2026-08-28,2026-09-15,29.9,EUR,paid,ORD-2026-2044-09
5,INV-2026-0906,2110,2026-08-28,2026-09-15,19.9,EUR,unpaid,ORD-2026-2110-09
6,INV-2026-0907,2235,2026-08-28,2026-09-15,49.9,EUR,paid,ORD-2026-2235-09
7,INV-2026-0908,2301,2026-08-28,2026-09-15,29.9,EUR,paid,ORD-2026-2301-09


`data["invoices"]` is already a list of flat records, so `pd.DataFrame(...)` is enough. If the payload were nested, we would use `json_normalize` as in the previous section.

If the local server were unavailable, the lesson would still run:

In [31]:
with open("data/raw/api_invoices.json", encoding="utf-8") as file:
    fallback = json.load(file)

pd.DataFrame(fallback["invoices"]).head()

,invoice_id,customer_id,invoice_date,due_date,amount,currency,status,order_id
0,INV-2026-0901,1827,2026-08-28,2026-09-15,49.9,EUR,unpaid,ORD-2026-1827-09
1,INV-2026-0902,1841,2026-08-28,2026-09-15,29.9,EUR,paid,ORD-2026-1841-09
2,INV-2026-0903,1903,2026-08-28,2026-09-15,19.9,EUR,paid,ORD-2026-1903-09
3,INV-2026-0904,1918,2026-08-28,2026-09-01,49.9,EUR,overdue,ORD-2026-1918-09
4,INV-2026-0905,2044,2026-08-28,2026-09-15,29.9,EUR,paid,ORD-2026-2044-09


That fallback is not a second dataset. It is the same HarborLink September invoices, stored as a file instead of being served over HTTP.

A public internet API can follow exactly the same pattern — `requests.get`, status code, `response.json()`, DataFrame — but a course that must remain reproducible should never depend on it.

### Other sources, mentioned only briefly

Web pages, HTML tables and PDF reports also appear in companies. HTML can be retrieved with `requests` and parsed with a library such as Beautiful Soup. We do not develop that path here. The priority of this lesson remains:

**CSV → Excel → JSON → SQL → API**

Those five routes already cover exports, workbooks, software payloads, operational databases and live interfaces.

### Part 2 synthesis

| Source or format | What it represents | Typical business use | Python approach |
| --- | --- | --- | --- |
| CSV | Tabular data | Exports, transactions, reports | `pandas.read_csv` |
| Excel | Spreadsheet data | Business reports, manual workflows | `pandas.read_excel` |
| JSON | Structured or semi-structured objects | APIs, software integration | `json`, `requests`, `pandas` |
| SQL database | Relational data | ERP / CRM / operational systems | SQL + connector + pandas |
| API | Interface to data or services | Software integration, external systems | `requests` or another client |
| ERP | Business information system | Finance, purchasing, sales, operations | Export, API, connector, ETL / ELT |
| CRM | Customer information system | Interactions, sales, support | Export, API, connector, ETL / ELT |

ERP and CRM are **source systems**. CSV, Excel and JSON are **representations or exchange formats**. SQL is a **query language**. An API is an **access interface**. Mixing these categories is one of the most common confusions in applied data work.

## Part 3 — Extracting data from documents: the PDF problem

The files in Part 2 already had a logical structure: columns, keys, tables, JSON fields. Documents are different.

> Extracting text from a document is not the same as reconstructing the structured information represented by that document.

### Why PDF extraction is difficult

A PDF is primarily a **document-preservation** format. It is designed to keep a page looking the same on another computer, another printer, another year. It is not designed to keep a table as a table.

A layout that looks like rows and columns to a human may internally be a collection of text fragments placed at coordinates. The consequences are practical:

- visual appearance and logical structure are different things;
- text may be stored in separate fragments;
- reading order may be ambiguous;
- columns may be interleaved;
- tables may have no explicit row or column metadata;
- headers and footers may be mixed into the extracted text;
- multi-column layouts may be read in the wrong order;
- page breaks may split a table;
- merged cells may be ambiguous;
- a scanned PDF may contain only images.

HarborLink's September invoice for Elena Vargas exists in two versions on purpose:

- `data/raw/invoice_text.pdf` — a text-based invoice;
- `data/raw/invoice_scanned.pdf` — a visually similar image-based invoice.

The business content is the same: invoice `INV-2026-0901`, customer 1827, premium subscription, 49.90 EUR, status unpaid. The encoding is not the same.

### Text-based PDFs

A text-based PDF contains machine-readable characters. We can open it and ask a library for the text and, sometimes, for tables.

We use **PyMuPDF**, imported as `pymupdf`. It can open a file, count pages, extract text, list images and try to detect tables.

In [32]:
import pymupdf

text_pdf = pymupdf.open("data/raw/invoice_text.pdf")
text_pdf.page_count

1

`pymupdf.open` opens the document. `page_count` is the number of pages. A page is a visual unit, not a database record.

In [33]:
page = text_pdf[0]
text = page.get_text()
print(text)

HarborLink BV  ·  Rue de la Science 14, 1040 Brussels, Belgium  ·  Confidential customer invoice
Page 1 of 1
HarborLink
INVOICE
INV-2026-0901
Customer-communication platform for small and mid-sized businesses
Bill to
Invoice details
Elena Vargas
Atelier Vega
Customer ID: 1827
BE
Invoice date: 2026-08-28
Due date: 2026-09-15
Order: ORD-2026-1827-09
Currency: EUR
Status: UNPAID
Description
Amount (EUR)
Premium subscription — September 2026
49.90
Total due
49.90
Notes
Payment for the September billing cycle has not been received as of 2026-09-12.



`text_pdf[0]` is the first page (Python starts counting at zero). `get_text()` returns the characters the library can read, in the order it reconstructs.

Look at the first lines. The **footer** of the invoice often appears before the title. That is not a HarborLink joke. It is evidence that reading order follows internal fragments, not necessarily the visual path of a human reader.

In [34]:
tables = page.find_tables()
invoice_tables = [table.to_pandas() for table in tables]
invoice_tables[0] if invoice_tables else "No table detected"

Consider using the pymupdf_layout package for a greatly improved page layout analysis.


,Description,Amount (EUR)
0,Premium subscription — September 2026,49.90
1,Total due,49.90


`find_tables()` looks for geometric arrangements that resemble tables. `to_pandas()` turns a detected table into a DataFrame. On this synthetic invoice the line items are recovered. That success is not a general guarantee. A slightly different layout, a wrapped description or a second page can break the reconstruction.

In [35]:
text_pdf.close()

### Scanned PDFs

A scanned PDF may contain a photograph of a page rather than machine-readable text. Ordinary text extraction then returns little or nothing useful.

In [36]:
scanned_pdf = pymupdf.open("data/raw/invoice_scanned.pdf")
scanned_page = scanned_pdf[0]

print("Characters extracted:", len(scanned_page.get_text() or ""))
print("Embedded images:", scanned_page.get_images())
print("Detected tables:", len(scanned_page.find_tables().tables))
scanned_pdf.close()

Characters extracted: 0
Embedded images: [(3, 0, 1240, 1754, 8, 'DeviceRGB', '', 'FormXob.7963aa8658f1ce182bee743e79239c3f', 'ASCII85Decode')]
Detected tables: 0


The scanned file looks like an invoice. Extraction finds an image and no usable text. The computer does not "see" 49.90 until another process turns the image into characters.

The conceptual chain for that other process is:

**Scanned PDF → image → OCR → text → structuring → DataFrame**

**OCR** (*optical character recognition*) is the step that guesses characters from pixels. It can be very useful. It can also misread a `5` as an `S`, invent a decimal, or skip a faded due date.

This lesson does **not** install an OCR engine. OCR typically needs extra system software and is not reliable enough to make a two-hour course depend on it. Treat the chain above as a concept you should know, not as a command you must run today.

### Tables and complex layouts

Invoices, financial reports and contracts concentrate the difficulties:

- tables that span several pages;
- multiple columns of narrative text;
- headers, footers and page numbers;
- footnotes;
- merged cells;
- nested tables;
- layouts that change from one month to the next.

Six jobs are often confused:

1. Extracting text.
2. Extracting geometric elements (boxes, lines, images).
3. Detecting tables.
4. Reconstructing rows and columns.
5. Interpreting the meaning of fields (`49.90` is an amount; `1827` is a customer identifier).
6. Validating the extracted data against another source.

A library that succeeds at step 1 may still fail at step 4. A model that succeeds at step 5 may still be wrong at step 6.

Compare the PDF amount with the database. If they disagree, the document extraction is not finished.

In [37]:
connection = sqlite3.connect("data/raw/company.sqlite")
check = pd.read_sql_query(
    """
    SELECT invoice_id, customer_id, amount, status, due_date
    FROM invoices
    WHERE invoice_id = 'INV-2026-0901'
    """,
    connection,
)
connection.close()
check

,invoice_id,customer_id,amount,status,due_date
0,INV-2026-0901,1827,49.9,unpaid,2026-09-15


The invoice, the JSON email, the customer CSV and this query all refer to customer 1827 and 49.90 EUR. That agreement is the beginning of validation.

### Modern document-extraction approaches

The current landscape, at a high level, includes:

- PDF parsers such as the one we just used;
- OCR for image-based pages;
- specialised table-extraction tools;
- document AI systems;
- cloud document-processing services;
- extraction based on large language models or vision-language models.

Modern AI approaches can help interpret complex documents. They do not remove the need for:

- validation;
- consistency checks;
- provenance (where did this number come from?);
- human review when the decision matters;
- confidentiality and security controls;
- explicit checks of numerical values, dates and identifiers.

We will not implement a complete LLM pipeline in this lesson. An impressive extraction that cannot be checked is not analysis-ready data.

### What the two PDFs were built to show

The text-based invoice contains real characters. The scanned invoice contains a picture of the same characters. Both were generated from the HarborLink scenario. If you open them side by side, they should look like ordinary business documents. The difference appears only when software tries to read them.

That is the PDF problem in one pair of files.

### Conclusion of Part 3

```{mermaid}
flowchart LR
    A[Document] --> B[Extraction]
    B --> C[Validation]
    C --> D[Structured fields]
    D --> E[DataFrame]
```

The difficulty is not merely reading characters. The challenge is recovering reliable structured information from a document whose layout and meaning may not be explicitly encoded.

## Synthesis

Business information is produced and stored in many heterogeneous forms: ERP systems, CRM systems, databases, spreadsheets, APIs, emails, tickets and documents. Data science begins by identifying these sources, retrieving the relevant information, extracting and validating its structure, and transforming it into reliable data that Python can analyse.

We can now close the loop with the HarborLink late-payment question. The useful analytical table did not come from a single magic command. It came from choosing a source, choosing an analytical unit, extracting a subset, and checking that identifiers and amounts still match the business story.

In [38]:
late

,customer_id,customer_name,company_name,invoice_id,amount,status
0,1827,Elena Vargas,Atelier Vega,INV-2026-0901,49.9,unpaid
1,2110,Jonas Keller,Keller Studio,INV-2026-0906,19.9,unpaid
2,1918,Thomas Berg,Berg Logistics,INV-2026-0904,49.9,overdue


The complete path, including the work of the next lesson, is:

**Heterogeneous business information
→ extraction and integration
→ structured data
→ pandas DataFrame
→ exploratory analysis and visualisation
→ statistical and machine-learning analysis**

### Towards next section

We now have rectangular tables: customers, invoices, payments, interactions. We have not yet described distributions, compared segments, or produced the plots that a business audience expects.

**The next section** will start where this lesson stops. The first task is to join the HarborLink extracts into one invoice-level analysis table. Only then will we use matplotlib and seaborn for exploratory analysis and visualisation.